In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "functions"))

In [2]:
import numpy as np
import torch
from model import EEG_CNN, ndarray_to_tensor, labels_to_tensor, get_device
from train import make_dataloader, train, evaluate, run_loso, run_loso_optimized
from dataset import get_loso_split, EEGDataset

# CNN Shape Test

In [3]:
model = EEG_CNN()
dummy = torch.randn(8, 22, 65, 125)
out = model(dummy)
print(f"CNN output shape: {out.shape}")  
# should be [8, 4]

CNN output shape: torch.Size([8, 4])


# DataLoader Test

In [4]:
X_dummy = np.random.randn(100, 22, 65, 125).astype(np.float32)
y_dummy = np.random.randint(0, 4, 100).astype(np.int64)
loader = make_dataloader(X_dummy, y_dummy, batch_size=16)
X_batch, y_batch = next(iter(loader))
print(f"batch X shape: {X_batch.shape}")  
# should be [16, 22, 65, 125]
print(f"batch y shape: {y_batch.shape}")  
# should be [16]

batch X shape: torch.Size([16, 22, 65, 125])
batch y shape: torch.Size([16])


# LOSO Test

In [5]:
X_dummy = np.random.randn(200, 22, 65, 125).astype(np.float32)
y_dummy = np.random.randint(0, 4, 200).astype(np.int64)
subjects = np.array([1]*100 + [2]*100)
accuracies, mean = run_loso(X_dummy, y_dummy, subjects, n_epochs=2, batch_size=16)
print(f"accuracies: {accuracies}")
print(f"mean: {mean}")  
# expect 0.25, rand chance -> 4 classes

1 16.666109613009862
2 8.498320307050433
1 0.27
1 20.384441563061305
2 13.390430314200264
2 0.19
accuracies: [0.27, 0.19]
mean: 0.23


# EEGDataset Test

In [3]:
# checks that EEGDataset loads correctly from disk and shapes are right
dataset = EEGDataset([1, 2], data_dir="../data/processed_per_subject")
print(f"Total samples: {len(dataset)}")
X_sample, y_sample = dataset[0]
print(f"Sample X shape: {X_sample.shape}")  
# should be [22, 65, 626]
print(f"Sample y: {y_sample}")  
# should be 0–3

Total samples: 576
Sample X shape: torch.Size([22, 65, 626])
Sample y: 3


/Users/ashvath/eeg-wheelchair-control/functions/dataset.py:243: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  x = torch.from_numpy(self.X_list[subject_idx][local_idx]).float()


# Run LOSO Optimized Test

In [5]:
# 2 epochs, 2 subjects, just checking it runs without errors
accuracies, mean = run_loso_optimized([1, 2], n_epochs=2, batch_size=16, data_dir="../data/processed_per_subject")
print(f"accuracies: {accuracies}")
print(f"mean: {mean}")  
# expect ~0.25, random chance -> 4 classes

[Subject 1] Epoch 1/2 | Loss: 2.8606
[Subject 1] Epoch 2/2 | Loss: 0.7359
[Subject 1] Accuracy: 0.2604

[Subject 2] Epoch 1/2 | Loss: 2.8011
[Subject 2] Epoch 2/2 | Loss: 0.8694
[Subject 2] Accuracy: 0.2431

Mean LOSO Accuracy: 0.2517
accuracies: [0.2604166666666667, 0.24305555555555555]
mean: 0.2517361111111111
